In [2]:
import sys
from pathlib import Path


def repo_root() -> Path:
    """Folder that contains ``src/`` (works from repo root, ``notebooks/``, or IDE cwd)."""
    cur = Path.cwd().resolve()
    for d in [cur, *cur.parents]:
        if (d / "src" / "Feature_Extraction.py").is_file():
            return d
    raise RuntimeError("Could not find project root (missing src/Feature_Extraction.py).")


PROJECT_ROOT = repo_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\pc\Desktop\Projects\Natural-Lang-Processing-Arabic\arabic-sentiment-analysis


In [3]:
data = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tweets_clean.csv")
data = data.dropna(subset=["clean_text", "label"]).reset_index(drop=True)

print("Shape:", data.shape)
print("Columns:", list(data.columns))
data.head()

Shape: (10001, 2)
Columns: ['clean_text', 'label']


,clean_text,label
0,بعد استقال رئيس محكم دستوري ننتظر استقال رئيس ...,OBJ
1,اهنئ دكتور احمد جمال دين قياد حزب مصر مناسب صد...,POS
2,برادع يستقو امريك مرهاخر و يرسل عصام عري الي ا...,NEG
3,حري عدال شاهد الان يله اتحادي اول يلم استقصائ ...,OBJ
4,والد لو اقول خاطر حشيش تضح بس من اقول ملل الل ...,NEUTRAL


In [ ]:
# Cell 1: shared prep (run once)
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"].astype(str),
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"],
)
MAX_WORDS, MAX_LEN = 20000, 120
tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(X_train)

Xtr = pad_sequences(tok.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post")
Xte = pad_sequences(tok.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding="post", truncating="post")

le = LabelEncoder()
ytr = le.fit_transform(y_train)
yte = le.transform(y_test)
NCLS = len(le.classes_)

In [ ]:
# Cell 2: LSTM
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(128),
    Dropout(0.3),
    Dense(NCLS, activation="softmax")
])

lstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
lstm.fit(Xtr, ytr, epochs=5, batch_size=64, validation_split=0.1, verbose=1)
lstm_acc = lstm.evaluate(Xte, yte, verbose=0)[1]
print("LSTM test acc:", lstm_acc)

In [ ]:
# Cell 3: CNN
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

cnn = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(NCLS, activation="softmax")
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(Xtr, ytr, epochs=5, batch_size=64, validation_split=0.1, verbose=1)
cnn_acc = cnn.evaluate(Xte, yte, verbose=0)[1]
print("CNN test acc:", cnn_acc)